# QLoRA fine-tune of the seller (RQ2, Agents C and D)

Trains a LoRA adapter on a 4-bit Qwen2.5-3B using `finetune/prepare_data.py`'s output, measures it
against the untuned model, and exports a GGUF for Ollama.

**On Colab:** Runtime > Change runtime type > pick a GPU, then Run all. Upload `train.jsonl` and `valid.jsonl`
from `data/craigslist/sft/` when asked. Outputs go to Google Drive, so a dropped session can resume: run all again.

## 1. Setup

In [ ]:
# Versions are pinned for reproducibility. Colab provides torch.
%pip install -q transformers==4.51.3 peft==0.15.2 accelerate==1.6.0 bitsandbytes==0.45.5 datasets==3.5.0 matplotlib

In [ ]:
import json, os, random, re, time
from pathlib import Path

import torch
from datasets import load_dataset
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, EarlyStoppingCallback,
                          Trainer, TrainingArguments, set_seed)
from transformers.trainer_utils import get_last_checkpoint

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"  # the weights behind Ollama's qwen2.5:3b, RQ5's weak model
REVISION = "aa8e72537993ba99e69dfaafa59ed015b17504d1"
SEED = 42
DATA_DIR = Path(os.environ.get("HAGGLE_DATA_DIR", "."))
DEFAULT_OUT = "out"
try:
    from google.colab import drive  # keep checkpoints where a disconnect can't lose them
    drive.mount("/content/drive")
    DEFAULT_OUT = "/content/drive/MyDrive/haggle_finetune"
except ImportError:
    pass
OUT = Path(os.environ.get("HAGGLE_OUT_DIR", DEFAULT_OUT))
MAX_STEPS = int(os.environ.get("HAGGLE_MAX_STEPS", -1))  # a short smoke test sets this; -1 trains fully
BATCH_SIZE = int(os.environ.get("HAGGLE_BATCH_SIZE", 4))  # 4 fits a 16 GB GPU; lower it if you run out of memory
GRAD_ACCUM = 8 // BATCH_SIZE  # keeps the effective batch at 8
EPOCHS = 2
EVAL_EVERY = int(os.environ.get("HAGGLE_EVAL_EVERY", 50))  # optimizer steps, about 16 checks over 2 passes
EVAL_EXAMPLES = int(os.environ.get("HAGGLE_EVAL_EXAMPLES", 100))  # a sample: scoring all of dev each time takes as long as training
MAX_LEN = 1536  # tokens; longer examples are dropped rather than cut, because a cut loses the target
N_CHECK = int(os.environ.get("HAGGLE_N_CHECK", 100))  # held-out examples for the before/after check

set_seed(SEED)
OUT.mkdir(parents=True, exist_ok=True)
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB")

In [ ]:
if not (DATA_DIR / "train.jsonl").exists():
    from google.colab import files
    files.upload()  # pick train.jsonl and valid.jsonl

## 2. Tokenise

Each row is one seller turn: the prompt the live agent would send, then the seller's real move.
Labels are `-100` over the prompt, so loss comes only from the final reply.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=REVISION)
tokenizer.padding_side = "right"
END = "<|im_end|>\n"

def encode(row):
    prompt = tokenizer.apply_chat_template(row["messages"][:-1], tokenize=False, add_generation_prompt=True)
    prompt_ids = tokenizer(prompt, add_special_tokens=False).input_ids
    target_ids = tokenizer(row["messages"][-1]["content"] + END, add_special_tokens=False).input_ids
    ids = prompt_ids + target_ids
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": [-100] * len(prompt_ids) + target_ids}

raw = load_dataset("json", data_files={"train": str(DATA_DIR / "train.jsonl"), "valid": str(DATA_DIR / "valid.jsonl")})
data = raw.map(encode, remove_columns=raw["train"].column_names).filter(lambda r: len(r["input_ids"]) <= MAX_LEN)
lengths = sorted(len(x) for x in data["train"]["input_ids"])
print({k: len(v) for k, v in data.items()}, "dropped", len(raw["train"]) - len(data["train"]),
      "median/max tokens", lengths[len(lengths) // 2], lengths[-1])

def collate(batch):
    longest = max(len(r["input_ids"]) for r in batch)
    pad = lambda r, key, value: r[key] + [value] * (longest - len(r[key]))
    return {
        "input_ids": torch.tensor([pad(r, "input_ids", tokenizer.pad_token_id) for r in batch]),
        "attention_mask": torch.tensor([pad(r, "attention_mask", 0) for r in batch]),
        "labels": torch.tensor([pad(r, "labels", -100) for r in batch]),
    }

# Eyeball one example: the decoded labels must be exactly the seller's JSON reply.
sample = data["train"][0]
print(tokenizer.decode([t for t in sample["labels"] if t != -100]))

## 3. Load the model in 4-bit and attach the LoRA adapter

In [ ]:
BF16 = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16 else torch.float16
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                           bnb_4bit_compute_dtype=DTYPE)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, revision=REVISION, quantization_config=quant, device_map={"": 0})
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                        gradient_checkpointing_kwargs={"use_reentrant": False})
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))
model.print_trainable_parameters()

## 4. Train

The best adapter by held-out loss is kept, and training stops early if that loss stops improving.

In [ ]:
args = TrainingArguments(
    output_dir=str(OUT / "checkpoints"), seed=SEED, data_seed=SEED, report_to="none",
    num_train_epochs=EPOCHS, max_steps=MAX_STEPS, learning_rate=2e-4, warmup_ratio=0.05, weight_decay=0.0,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
    bf16=BF16, fp16=not BF16, optim="adamw_torch", logging_steps=10,
    eval_strategy="steps", eval_steps=EVAL_EVERY, save_strategy="steps", save_steps=EVAL_EVERY, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    remove_unused_columns=False, label_names=["labels"],
)
trainer = Trainer(model=model, args=args, train_dataset=data["train"], eval_dataset=data["valid"].shuffle(seed=SEED).select(range(min(EVAL_EXAMPLES, len(data["valid"])))), data_collator=collate,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=4)])
started = time.time()
last_checkpoint = get_last_checkpoint(args.output_dir) if os.path.isdir(args.output_dir) else None
print("resuming from", last_checkpoint) if last_checkpoint else None
trainer.train(resume_from_checkpoint=last_checkpoint)
train_minutes = (time.time() - started) / 60
model.save_pretrained(OUT / "adapter")
tokenizer.save_pretrained(OUT / "adapter")
adapter_mb = sum(f.stat().st_size for f in (OUT / "adapter").glob("*.safetensors")) / 2**20
print(f"{train_minutes:.1f} min, adapter {adapter_mb:.0f} MB")

In [ ]:
import matplotlib.pyplot as plt

log = trainer.state.log_history
train_pts = [(x["step"], x["loss"]) for x in log if "loss" in x]
eval_pts = [(x["step"], x["eval_loss"]) for x in log if "eval_loss" in x]
plt.plot(*zip(*train_pts), label="training loss")
plt.plot(*zip(*eval_pts), marker="o", label="held-out loss")
plt.xlabel("optimizer step"); plt.ylabel("loss"); plt.legend(); plt.title("QLoRA fine-tune of the seller")
plt.savefig(OUT / "loss_curve.png", dpi=150); plt.show()
(OUT / "training_log.json").write_text(json.dumps(log, indent=1))

## 5. Tuned vs untuned on held-out turns

Generates **without** a JSON schema, so this shows whether the model learned the format itself. In real
games the schema is enforced, so this measures learning rather than live failures. It also checks moves
against what the good human seller did, and how often an offer breaks the floor in the prompt.

In [ ]:
ACTIONS = {"offer", "accept", "chat", "reject", "walk_away"}

def parse_reply(text):
    try:
        reply = json.loads(text)
    except json.JSONDecodeError:
        return None
    ok = (isinstance(reply, dict) and isinstance(reply.get("message"), str) and reply.get("action") in ACTIONS
          and (reply["action"] != "offer" or (isinstance(reply.get("price"), (int, float)) and reply["price"] > 0)))
    return reply if ok else None

def check(n):
    rows = random.Random(SEED).sample(list(raw["valid"]), min(n, len(raw["valid"])))
    valid = same_action = below_floor = offers = 0
    price_errors, seconds, new_tokens = [], 0.0, 0
    for row in rows:
        prompt = tokenizer.apply_chat_template(row["messages"][:-1], tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
        started = time.time()
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=200, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        seconds += time.time() - started
        new = out[0][inputs.input_ids.shape[1]:]
        new_tokens += len(new)
        reply = parse_reply(tokenizer.decode(new, skip_special_tokens=True).strip())
        if reply is None:
            continue
        valid += 1
        truth = json.loads(row["messages"][-1]["content"])
        floor = float(re.search(r"\(your floor\) is SGD ([\d.]+)\.", row["messages"][0]["content"]).group(1))
        same_action += reply["action"] == truth["action"]
        if reply["action"] == "offer":
            offers += 1
            below_floor += reply["price"] < floor
            if truth["action"] == "offer":
                price_errors.append(abs(reply["price"] - truth["price"]) / truth["price"])
    return {
        "examples": len(rows), "valid_json_rate": valid / len(rows), "same_action_as_human": same_action / max(valid, 1),
        "offers_below_floor": below_floor / max(offers, 1), "median_price_error": sorted(price_errors)[len(price_errors) // 2] if price_errors else None,
        "tokens_per_second": new_tokens / seconds,
    }

model.config.use_cache = True
model.gradient_checkpointing_disable()
model.eval()
with model.disable_adapter():
    untuned = check(N_CHECK)
tuned = check(N_CHECK)
report = {"base_model": BASE_MODEL, "revision": REVISION, "seed": SEED, "train_minutes": train_minutes, "adapter_mb": adapter_mb,
          "best_eval_loss": trainer.state.best_metric, "untuned": untuned, "tuned": tuned}
(OUT / "report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))

## 6. Export for Ollama

Merges the adapter into full-precision weights, converts them to GGUF and quantises to Q4_K_M, the same
format as Ollama's stock `qwen2.5:3b`. Download `qwen2.5-3b-haggle-q4_k_m.gguf` from the output folder, and put it next to
`finetune/Modelfile`.

In [ ]:
del model, trainer
torch.cuda.empty_cache()
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, revision=REVISION, torch_dtype=torch.float16)  # on the CPU: merging needs no GPU
merged = PeftModel.from_pretrained(base, OUT / "adapter").merge_and_unload()
merged.save_pretrained(OUT / "merged")
tokenizer.save_pretrained(OUT / "merged")

In [ ]:
%%bash -s "$OUT"
set -e
OUT=$1
[ -d llama.cpp ] || git clone -q --branch b6000 --depth 1 https://github.com/ggml-org/llama.cpp
pip install -q gguf==0.16.2 sentencepiece protobuf
python llama.cpp/convert_hf_to_gguf.py "$OUT/merged" --outtype f16 --outfile "$OUT/merged-f16.gguf"
cmake -S llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF > /dev/null
cmake --build llama.cpp/build --target llama-quantize -j > /dev/null
llama.cpp/build/bin/llama-quantize "$OUT/merged-f16.gguf" "$OUT/qwen2.5-3b-haggle-q4_k_m.gguf" q4_k_m
rm "$OUT/merged-f16.gguf"
ls -lh "$OUT"